# Retrain Pipeline — CV Structural Parser
**Kilani Groupe — Production retraining**

Déclenché automatiquement par GitHub Actions via n8n.
Utilise les feedbacks RH stockés dans PostgreSQL.

In [ ]:
# Paramètres injectés par papermill
data_path         = "ia_service/training_data/feedbacks.json"
model_output_path = "ia_service/models/structure_model.pkl"
min_f1_improvement = 0.01

In [ ]:
import os, re, json, warnings, time
from collections import Counter
from typing import Tuple

import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm

from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, VotingClassifier
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix
)
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# === CONSTANTES (identiques à cv_extractor.py et layer1) ===

BLOCK_TYPES = {
    'SECTION_HEADER': 0, 'JOB_TITLE': 1, 'COMPANY': 2,
    'BULLET_POINT':   3, 'DIPLOMA':   4, 'CONTACT': 5, 'OTHER': 6,
}
BLOCK_NAMES  = {v: k for k, v in BLOCK_TYPES.items()}
N_CLASSES    = len(BLOCK_TYPES)
CLASS_LABELS = [BLOCK_NAMES[i] for i in range(N_CLASSES)]

_EMAIL_RE = re.compile(
    r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}"
)
_DATE_RANGE_RE = re.compile(
    r'((?:19|20)\d{2})\s*[-\u2013\u2014/]\s*'
    r"((?:19|20)\d{2}|pr[eé]sent|actuel|current|now|ongoing)",
    re.IGNORECASE
)

SECTION_ANCHORS_FLAT = {
    'experience','experiences','education','formation','formations',
    'competences','skills','langues','languages','certifications',
    'profil','summary','profile','projets','projects','stage','stages',
    'diplomes','degrees','contact','about','associations','parcours',
    'work experience','professional experience','technical skills',
}

DIPLOMA_PATTERNS = [
    r"Ing[eé]nieur\s+(?:en\s+)?[\w\s\-']{2,50}",
    r"Licence\s+(?:en|de|Professionnelle)[\w\s\-']{0,40}",
    r"Master\s+(?:en|de|of|Professionnel)[\w\s\-']{0,40}",
    r"Bachelor\s+(?:of|in)\s+[\w\s\-']{2,40}",
    r"(?:MSc|MBA|PhD|BSc)\s+[\w\s\(\)\-']{2,40}",
    r"Doctorat\s+en\s+[\w\s\-']{2,40}",
]

COMPANY_PATTERNS = [
    r'\b(?:s\.?a\.?r\.?l\.?|s\.?a\.?s\.?|s\.?a\.?)\b',
    r'\b(?:group(?:e)?|holding|corp|inc\.?|ltd\.?)\b',
    r'\b(?:telecom|technologies?|solutions?|consulting|services?)\b',
]

print(f'Constants OK — {N_CLASSES} classes: {CLASS_LABELS}')

In [ ]:
# === FEATURE ENGINEERING (identique à cv_extractor.py et layer1 cellule 6) ===

def extract_structural_features(
    line: str, prev_line: str = '', next_line: str = '',
    position: int = 0, total_lines: int = 1,
) -> dict:
    lc    = line.strip()
    words = lc.split()
    alpha = [c for c in lc if c.isalpha()]
    upper_ratio = sum(1 for c in alpha if c.isupper()) / max(len(alpha), 1)
    digit_ratio = sum(1 for c in lc if c.isdigit()) / max(len(lc), 1)
    special     = sum(1 for c in lc if c in '@.:;/\\|+()[]{}#%&')
    char_var    = len(set(lc.lower())) / max(len(lc), 1)
    return {
        'length':           min(len(lc), 300),
        'word_count':       len(words),
        'avg_word_len':     round(np.mean([len(w) for w in words]) if words else 0, 2),
        'position_ratio':   round(position / max(total_lines, 1), 3),
        'is_first_5':       int(position < 5),
        'is_first_15':      int(position < 15),
        'is_top_20pct':     int(position < total_lines * 0.20),
        'is_bottom_20pct':  int(position > total_lines * 0.80),
        'is_all_upper':     int(bool(lc) and lc.isupper()),
        'is_title_case':    int(bool(lc) and lc.istitle()),
        'is_all_lower':     int(bool(lc) and lc.islower()),
        'upper_ratio':      round(upper_ratio, 2),
        'starts_upper':     int(bool(lc) and lc[0].isupper()),
        'is_short':         int(len(lc) < 20),
        'is_medium':        int(20 <= len(lc) < 60),
        'is_long':          int(len(lc) >= 60),
        'is_very_long':     int(len(lc) >= 120),
        'has_colon':        int(':' in lc),
        'has_pipe':         int('|' in lc),
        'has_slash':        int('/' in lc),
        'has_dash':         int('-' in lc or '\u2013' in lc),
        'has_comma':        int(',' in lc),
        'has_dot':          int('.' in lc),
        'has_parentheses':  int('(' in lc and ')' in lc),
        'has_at_sign':      int('@' in lc),
        'has_plus':         int('+' in lc),
        'special_char_count': min(special, 10),
        'is_bullet_start':  int(lc[:1] in '\u2022-*\u25aa\u25b8\xb7\u2013'
                                 or bool(re.match(r'^\d+[.)]\s', lc))),
        'bullet_char':      int(lc[:1] in '\u2022\u25aa\u25b8\xb7'),
        'has_digit':        int(any(c.isdigit() for c in lc)),
        'digit_ratio':      round(digit_ratio, 2),
        'has_4digit_seq':   int(bool(re.search(r'\d{4}', lc))),
        'has_date_sep':     int(bool(_DATE_RANGE_RE.search(lc))),
        'pure_number':      int(bool(re.match(r'^\d+$', lc))),
        'is_separator':     int(bool(re.match(r'^[-_=*\\.]{3,}$', lc))),
        'is_empty_ish':     int(len(lc) <= 2),
        'prev_empty':       int(len(prev_line.strip()) == 0),
        'prev_length':      min(len(prev_line.strip()), 200),
        'prev_is_upper':    int(bool(prev_line.strip()) and prev_line.strip().isupper()),
        'next_empty':       int(len(next_line.strip()) == 0),
        'next_length':      min(len(next_line.strip()), 200),
        'next_is_bullet':   int(next_line.strip()[:1] in '\u2022-*\u25aa\u25b8\xb7'),
        'surrounded_empty': int(not prev_line.strip() and not next_line.strip()),
        'contains_at':      int('@' in lc),
        'contains_www':     int('www' in lc.lower() or 'http' in lc.lower()),
        'word_count_small': int(1 <= len(words) <= 4),
        'is_md_h1':         int(lc.startswith('# ') and not lc.startswith('## ')),
        'is_md_h2':         int(lc.startswith('## ') and not lc.startswith('### ')),
        'is_md_h3':         int(lc.startswith('### ')),
        'is_md_bullet':     int(lc.startswith('- ') or lc.startswith('* ')),
        'is_md_hr':         int(lc == '---' or lc == '==='),
        'has_url':          int(bool(re.search(r'https?://|www\.', lc, re.I))),
        'starts_verb':      int(bool(re.match(r'^(?:develop|manag|design|implem|lead|build|creat|analys)', lc, re.I))),
        'has_semicolon':    int(';' in lc),
        'ends_colon':       int(lc.endswith(':')),
        'is_email_like':    int(bool(_EMAIL_RE.search(lc))),
        'is_phone_like':    int(bool(re.search(r'\+?\d[\d\s\-\.]{7,}', lc))),
        'char_variety':     round(char_var, 3),
    }

FEATURE_NAMES = list(extract_structural_features('test').keys())
print(f'Feature Engineering OK — {len(FEATURE_NAMES)} features')

In [ ]:
# === ANNOTATEUR HEURISTIQUE (identique à layer1 cellule 7) ===

def annotate_line(line: str, prev: str, nxt: str) -> int:
    lc = line.strip()
    ll = lc.lower()
    if not lc or len(lc) < 2:                    return BLOCK_TYPES['OTHER']
    if re.match(r'^#{1,3}\s+', lc):              return BLOCK_TYPES['SECTION_HEADER']
    if re.match(r'^[-*]\s+\w', lc):              return BLOCK_TYPES['BULLET_POINT']
    if _EMAIL_RE.search(lc):                      return BLOCK_TYPES['CONTACT']
    if re.match(r'^\+?\d[\d\s.\-()]{8,}$', lc): return BLOCK_TYPES['CONTACT']
    if re.match(r'^[-_=*\.]{3,}$', lc):          return BLOCK_TYPES['OTHER']
    ll_clean = ll.strip(':').strip()
    if ll_clean in SECTION_ANCHORS_FLAT or any(
        ll_clean.startswith(a) for a in SECTION_ANCHORS_FLAT if len(a) > 5
    ):
        return BLOCK_TYPES['SECTION_HEADER']
    for pat in DIPLOMA_PATTERNS:
        if re.search(pat, lc, re.I): return BLOCK_TYPES['DIPLOMA']
    if lc[:1] in '\u2022\u25aa\u25b8\xb7' or re.match(r'^\d+[.)]\s+\w', lc):
        return BLOCK_TYPES['BULLET_POINT']
    if any(re.search(p, ll, re.I) for p in COMPANY_PATTERNS) and len(lc) < 70:
        return BLOCK_TYPES['COMPANY']
    words = lc.split()
    if 2 <= len(words) <= 5 and not re.search(r'[\d@|]', lc) and lc[0].isupper():
        return BLOCK_TYPES['JOB_TITLE']
    return BLOCK_TYPES['OTHER']

print('Annotator OK')

In [ ]:
# === CHARGEMENT DES FEEDBACKS POSTGRESQL ===

with open(data_path, encoding='utf-8') as f:
    feedbacks = json.load(f)

print(f'{len(feedbacks)} feedbacks chargés depuis {data_path}')

rows, labels = [], []

for fb in tqdm(feedbacks, desc='Extraction features'):
    cv = fb.get('cv_parsed', {})

    # Récupérer le texte brut du CV parsé
    text = ''
    if isinstance(cv, dict):
        text = cv.get('raw_text', '') or cv.get('texte_brut', '')
        if not text:
            # fallback : concaténer toutes les valeurs string
            text = ' '.join(str(v) for v in cv.values() if isinstance(v, str) and v)
    elif isinstance(cv, str):
        text = cv

    if not text or len(text) < 10:
        continue

    lines = text.split('\n')
    total = len(lines)

    for i, line in enumerate(lines):
        ls = line.strip()
        if not ls or len(ls) < 2:
            continue
        prev  = lines[i-1].strip() if i > 0 else ''
        nxt   = lines[i+1].strip() if i < total-1 else ''
        feats = extract_structural_features(ls, prev, nxt, i, total)
        label = annotate_line(ls, prev, nxt)
        rows.append(list(feats.values()))
        labels.append(label)

X_raw = np.array(rows)
y_raw = np.array(labels)

data_disponible = len(X_raw) > 0

if not data_disponible:
    print('\nAUCUNE DONNEE EXPLOITABLE dans les feedbacks fournis '
          '(feedbacks.json vide ou textes trop courts/non parsables). '
          'Entrainement ignore, aucun modele ne sera promu.')
else:
    print(f'\nDataset brut : {len(X_raw):,} lignes')
    print('Distribution des classes :')
    for cid, cnt in sorted(Counter(y_raw).items()):
        print(f'  {BLOCK_NAMES[cid]:<18}: {cnt:>5,} ({cnt/len(y_raw)*100:.1f}%)')

In [ ]:
# === EQUILIBRAGE DES CLASSES (identique a layer1 cellule 12) ===

def balance_classes(X, y, target_ratio=0.25):
    counts = Counter(y)
    majority_count = max(counts.values())
    X_parts, y_parts = [X], [y]
    rng = np.random.default_rng(42)
    for cls, cnt in counts.items():
        ratio  = 0.50 if cls == BLOCK_TYPES['DIPLOMA'] else target_ratio
        target = max(int(majority_count * ratio), cnt * 3)
        if cnt < target:
            idx   = np.where(y == cls)[0]
            extra = rng.choice(idx, size=target - cnt, replace=True)
            X_parts.append(X[extra])
            y_parts.append(np.full(target - cnt, cls))
    X_bal = np.vstack(X_parts)
    y_bal = np.concatenate(y_parts)
    perm  = rng.permutation(len(X_bal))
    return X_bal[perm], y_bal[perm]

if data_disponible:
    X_bal, y_bal = balance_classes(X_raw, y_raw)
    print(f'Lignes equilibrees : {len(X_bal):,}')

    X_df = pd.DataFrame(X_bal, columns=FEATURE_NAMES)
    X_train, X_test, y_train, y_test = train_test_split(
        X_df, y_bal, test_size=0.20, random_state=42, stratify=y_bal
    )
    print(f'Split : Train={len(X_train):,} | Test={len(X_test):,}')
else:
    X_train = X_test = pd.DataFrame(columns=FEATURE_NAMES)
    y_train = y_test  = np.array([])
    print('Split ignore (aucune donnee disponible).')


In [ ]:
# === DÉFINITION DES MODÈLES (identique à layer1 cellule 16) ===

MODELS = {
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=30,
        max_features='sqrt', class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.08,
        subsample=0.65, colsample_bytree=0.75,
        min_child_samples=30, class_weight='balanced',
        reg_alpha=0.1, reg_lambda=0.1,
        random_state=42, verbose=-1
    ),
    'Voting Ensemble (RF + ExtraTrees + GBM)': VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(
                n_estimators=300, max_depth=10, min_samples_leaf=30,
                max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1
            )),
            ('et', ExtraTreesClassifier(
                n_estimators=200, max_depth=8, min_samples_leaf=35,
                max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1
            )),
            ('gb', GradientBoostingClassifier(
                n_estimators=150, max_depth=3, learning_rate=0.10,
                subsample=0.6, min_samples_leaf=30, random_state=42
            )),
        ],
        voting='soft', n_jobs=-1
    ),
}

MODEL_NAMES_SHORT = {
    'Random Forest':                           'Random Forest',
    'LightGBM':                                'LightGBM',
    'Voting Ensemble (RF + ExtraTrees + GBM)': 'Voting Ensemble',
}

print('Modèles définis :')
for name in MODELS:
    print(f'  - {name}')

In [ ]:
# === ENTRAINEMENT & EVALUATION (identique a layer1 cellule 17) ===

results = {}

if not data_disponible:
    print("Entrainement ignore (aucune donnee disponible).")
else:
    min_class_count = min(Counter(y_train).values())
    n_splits_safe   = max(2, min(5, min_class_count))
    cv_possible     = len(X_train) >= 4 and min_class_count >= 2

    if not cv_possible:
        print(f"ATTENTION : trop peu de donnees pour la cross-validation "
              f"(train={len(X_train)}, classe la plus rare={min_class_count}). "
              f"CV ignoree, entrainement direct sur train/test uniquement.")
    else:
        cv_strategy = StratifiedKFold(n_splits=n_splits_safe, shuffle=True, random_state=42)
        if n_splits_safe < 5:
            print(f"Note : n_splits reduit a {n_splits_safe} (classe la plus rare={min_class_count}).")

    for model_name, model in MODELS.items():
        short = MODEL_NAMES_SHORT[model_name]
        print(f'\n{"-"*55}')
        print(f'Entrainement : {short}')
        print(f'{"-"*55}')

        if cv_possible:
            t0 = time.perf_counter()
            cv_scores = cross_val_score(
                model, X_train, y_train,
                cv=cv_strategy, scoring='f1_weighted', n_jobs=-1
            )
        else:
            cv_scores = np.array([np.nan])

        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        fit_time = time.perf_counter() - t0

        y_pred    = model.predict(X_test)
        train_acc = model.score(X_train, y_train)
        test_acc  = accuracy_score(y_test, y_pred)
        f1_w      = f1_score(y_test, y_pred, average='weighted')
        f1_mac    = f1_score(y_test, y_pred, average='macro')
        prec_w    = precision_score(y_test, y_pred, average='weighted')
        rec_w     = recall_score(y_test, y_pred, average='weighted')
        gap       = train_acc - test_acc
        f1_per_class = f1_score(y_test, y_pred, average=None, labels=list(range(N_CLASSES)))
        cm = confusion_matrix(y_test, y_pred, labels=list(range(N_CLASSES)))

        results[model_name] = {
            'model':           model,
            'short_name':      short,
            'cv_scores':       cv_scores,
            'cv_mean':         np.nanmean(cv_scores),
            'cv_std':          np.nanstd(cv_scores),
            'train_acc':       train_acc,
            'test_acc':        test_acc,
            'f1_weighted':     f1_w,
            'f1_macro':        f1_mac,
            'precision_w':     prec_w,
            'recall_w':        rec_w,
            'overfitting_gap': gap,
            'fit_time':        fit_time,
            'f1_per_class':    f1_per_class,
            'cm':              cm,
            'y_pred':          y_pred,
        }

        print(f'  CV F1 ({n_splits_safe if cv_possible else 0}-fold) : {np.nanmean(cv_scores):.4f} +/- {np.nanstd(cv_scores):.4f}')
        print(f'  Train Accuracy  : {train_acc:.4f}')
        print(f'  Test  Accuracy  : {test_acc:.4f}')
        print(f'  F1 Weighted     : {f1_w:.4f}')
        print(f'  Overfitting Gap : {gap:.4f} ({"WARNING" if gap > 0.05 else "OK"})')
        print(f'  Fit time        : {fit_time:.1f}s')

    print('\nTous les modeles entraines.')


In [ ]:
# === SELECTION DU MEILLEUR MODELE + COMPARAISON + EXPORT CONDITIONNEL ===
# (base sur layer1 cellules 41 + 42, avec comparaison vs ancien modele)

MIN_TRAIN_SAMPLES_FOR_PROMOTION = 30

if not data_disponible or not results:
    print("=" * 60)
    print("AUCUNE DONNEE / AUCUN RESULTAT — pas d'entrainement effectue.")
    print("=" * 60)

    retrain_result = {
        'promoted': False,
        'new_f1':   0.0,
        'old_f1':   0.0,
        'model':    None,
        'reason':   'no_data',
    }

else:
    # Selectionner le meilleur modele (meme logique que layer1)
    best_model_name = max(results, key=lambda k: results[k]['f1_weighted'])
    best_result     = results[best_model_name]
    new_model       = best_result['model']
    new_f1          = best_result['f1_weighted']

    print('=' * 60)
    print('SELECTION DU MEILLEUR MODELE')
    print('=' * 60)
    print(f'Modele retenu  : {best_result["short_name"]}')
    print(f'F1 Weighted    : {new_f1:.4f}')
    print(f'Test Accuracy  : {best_result["test_acc"]:.4f}')
    print(f'CV F1          : {best_result["cv_mean"]:.4f} +/- {best_result["cv_std"]:.4f}')
    print()
    print('Classement complet :')
    ranked = sorted(results.items(), key=lambda x: x[1]['f1_weighted'], reverse=True)
    for rank, (name, res) in enumerate(ranked, 1):
        marker = ' <- RETENU' if name == best_model_name else ''
        print(f'  {rank}. {res["short_name"]:<30} F1={res["f1_weighted"]:.4f}{marker}')

    # Comparer avec l'ancien modele en production
    old_f1 = 0.0
    if os.path.exists(model_output_path):
        old_model = joblib.load(model_output_path)
        old_f1 = f1_score(
            y_test,
            old_model.predict(X_test),
            average='weighted'
        )
        print(f'\nAncien modele F1 : {old_f1:.4f}')
    else:
        print('\nAucun modele existant — premier deploiement.')

    print(f'Nouveau modele F1 : {new_f1:.4f}')
    print(f'Amelioration      : {new_f1 - old_f1:+.4f} (seuil requis : +{min_f1_improvement})')

    # Garde-fou : ne jamais promouvoir un modele entraine sur un dataset trop
    # petit, meme si le F1 calcule semble bon (resultat non fiable).
    data_insuffisante = len(X_train) < MIN_TRAIN_SAMPLES_FOR_PROMOTION
    if data_insuffisante:
        print(f"\nDonnees insuffisantes pour promotion (train={len(X_train)} < "
              f"{MIN_TRAIN_SAMPLES_FOR_PROMOTION}). Le meilleur modele est loggue "
              f"mais PAS promu, quel que soit son F1.")

    retrain_result = {
        'promoted': False,
        'new_f1':   round(float(new_f1), 4),
        'old_f1':   round(float(old_f1), 4),
        'model':    best_result['short_name'],
    }

    if (not data_insuffisante) and new_f1 >= old_f1 + min_f1_improvement:
        os.makedirs(os.path.dirname(model_output_path), exist_ok=True)
        joblib.dump(new_model, model_output_path, compress=3)

        metadata = {
            'model_name':    best_result['short_name'],
            'model_class':   type(new_model).__name__,
            'n_features':    len(FEATURE_NAMES),
            'feature_names': FEATURE_NAMES,
            'n_classes':     N_CLASSES,
            'class_labels':  CLASS_LABELS,
            'metrics': {
                'f1_weighted':     round(float(best_result['f1_weighted']), 4),
                'f1_macro':        round(float(best_result['f1_macro']), 4),
                'test_accuracy':   round(float(best_result['test_acc']), 4),
                'cv_mean':         round(float(best_result['cv_mean']), 4),
                'cv_std':          round(float(best_result['cv_std']), 4),
                'overfitting_gap': round(float(best_result['overfitting_gap']), 4),
            },
            'training_info': {
                'train_size':      int(len(X_train)),
                'test_size':       int(len(X_test)),
                'n_feedbacks':     int(len(feedbacks)),
                'n_classes':       N_CLASSES,
            }
        }
        meta_path = model_output_path.replace('.pkl', '_metadata.json')
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

        retrain_result['promoted'] = True
        print(f'\nPROMU — modele remplace : {model_output_path}')
        print(f'Metadonnees             : {meta_path}')
    else:
        print('\nNON PROMU — amelioration insuffisante ou donnees insuffisantes, modele inchange.')

# Ecrire le resultat pour GitHub Actions (toujours, meme sans donnees)
with open('retrain_result.json', 'w') as f:
    json.dump(retrain_result, f)

print(f'\nResultat : {retrain_result}')
